# Module 1.5 — RAG Evaluation Inside the Pipeline

Modules 1.1–1.4 all scored a RAG system's output *after the fact* — build a test case, hand it to a metric, read the score. That's how you evaluate offline, against a batch of test cases. But context relevance and faithfulness are exactly the kind of checks you can also run **as part of the pipeline itself**, gating generation on bad retrieval or flagging a hallucinated answer before it ever reaches a user. This notebook builds that as a small LangGraph — `retrieve → judge_context_relevance → generate → check_faithfulness` — first with deterministic mocks (no API key needed), then with the exact same graph shape wired to real LLM calls.

_Source: adapted from `04_Agent_RAG_Eval/rag_agent_eval_langgraph_new.ipynb` Part 2 (mock) and `rag_agent_eval_langgraph_openai.ipynb` Part 2 (real), lightly reframed to connect explicitly to Modules 1.2–1.3's metric vocabulary.

## Why wire evaluation into the graph at all?

Two reasons this pattern is worth knowing, beyond "it's a different way to compute the same numbers":

1. **You can act on the result mid-pipeline.** `judge_context_relevance` runs *before* generation — cheap enough, and early enough, that a real system could branch on it (skip generation and ask a clarifying question if nothing relevant was retrieved, instead of generating a plausible-sounding answer from noise).
2. **It's the same two ideas from Module 1.2 (Contextual Relevancy) and Module 1.3 (Faithfulness), computed by hand.** Seeing "context relevance" and "faithfulness" built from raw judge calls first, before treating them as black-box DeepEval metrics, makes it much clearer what those metrics are actually doing under the hood — and where you'd reach for a hand-rolled node instead of a metric class (when you need the eval signal *inside* your production control flow, not just in an offline report).

## Part A — Mocked: deterministic judges, no API key

In [ ]:
# ============ IMPORTS ============
from typing import TypedDict, List, Dict
from langgraph.graph import StateGraph, END

print("Imports OK")

In [ ]:
# ============ STATE SCHEMA ============
class RAGEvalState(TypedDict):
    query: str
    retrieved_chunks: List[Dict]
    relevance_judgments: Dict[str, str]     # chunk_id -> "relevant" / "irrelevant"
    answer: str
    claims: List[str]
    faithfulness_scores: Dict[str, bool]    # claim -> is it supported by context?
    context_relevance_score: float
    faithfulness_score: float

In [ ]:
# ============ MOCK JUDGES ============
# In production these would be real LLM calls (e.g. "Is this chunk relevant to
# the query? yes/no", or "Is this claim entailed by the context? yes/no").
STOPWORDS = {"what", "is", "the", "of", "a", "an", "to", "in", "for", "and", "how"}


def mock_judge_relevance(query: str, chunk_text: str) -> str:
    query_terms = {w for w in query.lower().split() if w not in STOPWORDS and len(w) > 2}
    overlap = sum(1 for term in query_terms if term in chunk_text.lower())
    return "relevant" if overlap >= 1 else "irrelevant"


def mock_generate_answer(query: str, chunks: List[Dict]) -> str:
    relevant_text = " ".join(c["text"] for c in chunks)
    return f"Based on context: {relevant_text[:60]}..."


def mock_extract_claims(answer: str) -> List[str]:
    return [s.strip() for s in answer.split(".") if s.strip()]


def mock_check_claim_supported(claim: str, chunks: List[Dict]) -> bool:
    all_text = " ".join(c["text"] for c in chunks).lower()
    key_terms = [w for w in claim.lower().split() if len(w) > 4]
    return any(term in all_text for term in key_terms)

print("Mock judges defined")

In [ ]:
# ============ GRAPH NODES ============
def retrieve_node(state: RAGEvalState) -> RAGEvalState:
    # In a real pipeline this is a vector search call. Fixture data here for a
    # reproducible demo -- notice c3 is deliberately off-topic.
    state["retrieved_chunks"] = [
        {"id": "c1", "text": "BST delete runs in O(h) time where h is tree height."},
        {"id": "c2", "text": "The successor node is the leftmost node of the right subtree."},
        {"id": "c3", "text": "Unrelated: Python lists are dynamic arrays."},
    ]
    return state


def judge_context_relevance_node(state: RAGEvalState) -> RAGEvalState:
    judgments = {c["id"]: mock_judge_relevance(state["query"], c["text"])
                 for c in state["retrieved_chunks"]}
    state["relevance_judgments"] = judgments
    relevant_count = sum(1 for v in judgments.values() if v == "relevant")
    state["context_relevance_score"] = relevant_count / len(judgments)
    return state


def generate_node(state: RAGEvalState) -> RAGEvalState:
    # Only feed relevant chunks into generation -- filter before you generate,
    # don't just filter the eval after the fact.
    relevant_chunks = [c for c in state["retrieved_chunks"]
                        if state["relevance_judgments"][c["id"]] == "relevant"]
    state["answer"] = mock_generate_answer(state["query"], relevant_chunks)
    return state


def check_faithfulness_node(state: RAGEvalState) -> RAGEvalState:
    claims = mock_extract_claims(state["answer"])
    state["claims"] = claims
    scores = {c: mock_check_claim_supported(c, state["retrieved_chunks"]) for c in claims}
    state["faithfulness_scores"] = scores
    supported = sum(1 for v in scores.values() if v)
    state["faithfulness_score"] = supported / len(scores) if scores else 0.0
    return state

print("Nodes defined")

In [ ]:
# ============ BUILD & RUN THE GRAPH ============
builder = StateGraph(RAGEvalState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("judge_context_relevance", judge_context_relevance_node)
builder.add_node("generate", generate_node)
builder.add_node("check_faithfulness", check_faithfulness_node)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "judge_context_relevance")
builder.add_edge("judge_context_relevance", "generate")
builder.add_edge("generate", "check_faithfulness")
builder.add_edge("check_faithfulness", END)

rag_eval_graph = builder.compile()

result = rag_eval_graph.invoke({"query": "What is the time complexity of BST delete?"})

print("Context relevance judgments:", result["relevance_judgments"])
print(f"Context relevance score:      {result['context_relevance_score']:.2f}")
print()
print("Answer:", result["answer"])
print("Faithfulness scores:", result["faithfulness_scores"])
print(f"Faithfulness score:           {result['faithfulness_score']:.2f}")

**Reading the output:**
- `c3` (Python lists — off-topic) is correctly flagged `irrelevant` → this is the **context relevance** signal (Module 1.2's Contextual Relevancy, computed by hand), checked *before* generation even happens.
- The **faithfulness score** (Module 1.3's Faithfulness, computed by hand) checks the *opposite direction*: given what got generated, is every claim traceable back to the retrieved context? A score of 1.00 means no hallucination was detected.
- These two catch **different failure modes**: bad context relevance = retriever problem. Bad faithfulness despite good context = generator problem (model ignored context / made things up) — the exact same distinction Module 1.3 drew between metrics, now visible as two separate *nodes* in a graph rather than two separate metric objects.

## Part B — Real: the same graph, wired to live `ChatOpenAI` calls

In [ ]:
# ============ REAL LLM SETUP ============
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY (shell env or .env) before running this cell."

MODEL_NAME = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)  # temperature=0: eval judges should be as deterministic as possible

print(f"OpenAI client ready -- model={MODEL_NAME}")

In [ ]:
# ============ REAL LLM-AS-JUDGE, VIA STRUCTURED OUTPUT ============
class RelevanceVerdict(BaseModel):
    verdict: str = Field(description='Either "relevant" or "irrelevant"')


class FaithfulnessVerdict(BaseModel):
    supported: bool = Field(description="True if the claim is entailed by the provided context")


class ExtractedClaims(BaseModel):
    claims: List[str] = Field(description="Atomic factual claims made in the answer, one per list item")


relevance_judge = llm.with_structured_output(RelevanceVerdict)
faithfulness_judge = llm.with_structured_output(FaithfulnessVerdict)
claim_extractor = llm.with_structured_output(ExtractedClaims)


def llm_judge_relevance(query: str, chunk_text: str) -> str:
    result = relevance_judge.invoke([
        SystemMessage("You judge whether a retrieved passage is relevant to a query. "
                       'Answer with exactly "relevant" or "irrelevant".'),
        HumanMessage(f"Query: {query}\n\nPassage: {chunk_text}"),
    ])
    return result.verdict.strip().lower()


def llm_generate_answer(query: str, chunks: List[Dict]) -> str:
    context = "\n".join(f"- {c['text']}" for c in chunks)
    result = llm.invoke([
        SystemMessage("Answer the question using ONLY the provided context. Be concise (1-2 sentences). "
                       "If the context is insufficient, say so."),
        HumanMessage(f"Context:\n{context}\n\nQuestion: {query}"),
    ])
    return result.content


def llm_extract_claims(answer: str) -> List[str]:
    result = claim_extractor.invoke([
        SystemMessage("Split the answer into a list of atomic, independently-checkable factual claims."),
        HumanMessage(answer),
    ])
    return result.claims


def llm_check_claim_supported(claim: str, chunks: List[Dict]) -> bool:
    context = "\n".join(f"- {c['text']}" for c in chunks)
    result = faithfulness_judge.invoke([
        SystemMessage("You check whether a claim is fully supported (entailed) by the given context. "
                       "Do not use outside knowledge."),
        HumanMessage(f"Context:\n{context}\n\nClaim: {claim}"),
    ])
    return result.supported

print("Real LLM judges defined")

Notice the *shape* of every function here is identical to Part A's mocks — same signature, same role in the graph — only the implementation swapped from string matching to a real judge call, using `with_structured_output` so each judge returns a typed verdict object instead of free text you'd have to parse yourself.

In [ ]:
# ============ SAME GRAPH SHAPE, REAL NODES ============
def retrieve_node(state: RAGEvalState) -> RAGEvalState:
    # Same fixture retrieval as Part A, for a direct mock-vs-real comparison.
    state["retrieved_chunks"] = [
        {"id": "c1", "text": "BST delete runs in O(h) time where h is tree height."},
        {"id": "c2", "text": "The successor node is the leftmost node of the right subtree."},
        {"id": "c3", "text": "Unrelated: Python lists are dynamic arrays."},
    ]
    return state


def judge_context_relevance_node(state: RAGEvalState) -> RAGEvalState:
    judgments = {
        c["id"]: llm_judge_relevance(state["query"], c["text"])
        for c in state["retrieved_chunks"]
    }
    state["relevance_judgments"] = judgments
    relevant_count = sum(1 for v in judgments.values() if v == "relevant")
    state["context_relevance_score"] = relevant_count / len(judgments)
    return state


def generate_node(state: RAGEvalState) -> RAGEvalState:
    relevant_chunks = [
        c for c in state["retrieved_chunks"]
        if state["relevance_judgments"][c["id"]] == "relevant"
    ]
    state["answer"] = llm_generate_answer(state["query"], relevant_chunks)
    return state


def check_faithfulness_node(state: RAGEvalState) -> RAGEvalState:
    claims = llm_extract_claims(state["answer"])
    state["claims"] = claims
    scores = {c: llm_check_claim_supported(c, state["retrieved_chunks"]) for c in claims}
    state["faithfulness_scores"] = scores
    supported = sum(1 for v in scores.values() if v)
    state["faithfulness_score"] = supported / len(scores) if scores else 0.0
    return state


builder = StateGraph(RAGEvalState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("judge_context_relevance", judge_context_relevance_node)
builder.add_node("generate", generate_node)
builder.add_node("check_faithfulness", check_faithfulness_node)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "judge_context_relevance")
builder.add_edge("judge_context_relevance", "generate")
builder.add_edge("generate", "check_faithfulness")
builder.add_edge("check_faithfulness", END)

rag_eval_graph = builder.compile()

result = rag_eval_graph.invoke({"query": "What is the time complexity of BST delete?"})

print("Context relevance judgments:", result["relevance_judgments"])
print(f"Context relevance score:      {result['context_relevance_score']:.2f}")
print()
print("Answer:", result["answer"])
print("Claims:", result["claims"])
print("Faithfulness scores:", result["faithfulness_scores"])
print(f"Faithfulness score:           {result['faithfulness_score']:.2f}")

**Reading the output:** unlike Part A, the exact judgments and claim wording here can vary slightly run-to-run because a real model is making the call — this is Module 0 §3's non-determinism caveat, made concrete. It's also why production evals typically run each judge multiple times, use `temperature=0` (already set above), and sometimes track judge-agreement itself as a metric rather than trusting a single pass.

## Summary

- The same two ideas taught as metrics in Modules 1.2–1.3 (Contextual Relevancy, Faithfulness) can be built directly into a pipeline's control flow as graph nodes — not just computed after the fact on a finished test case.
- Doing this by hand once (Part A) is what makes it clear what a metric class like `ContextualRelevancyMetric` is actually doing when you use it as a black box later.
- Swapping mock judges for real ones (Part B) changes exactly one thing — the judge implementation — while the graph shape, state schema, and node responsibilities stay identical. That's a useful property when prototyping: build and debug the graph shape against free, deterministic mocks, then swap in real judges once the shape is right.
- Next: [Module 1.6](06_RAGAS_in_Practice.ipynb) looks at RAGAS as a second implementation of these same generator-side ideas; [Module 1.7](07_RAG_Capstone_Build_and_Evaluate.ipynb) is the full capstone — a real RAG system, a synthetic golden dataset, and the complete metric suite run end to end.